# NB-01-01 — Capability Test Bench

This notebook turns the definitions in M01 into measurements you can run.

**What you will do**

1. Score described systems against a four-flag capability test.
2. Measure how often the field produced a public milestone, era by era.
3. Compare a rule-based and a learned classifier on the same six emails.
4. Check whether a model's stated confidence matches its accuracy.
5. See how per-step reliability compounds over a multi-step plan.

**Before you start:** you need Python 3.10+ and the standard library only, except for
the two plotting cells which use Matplotlib. Nothing here needs a GPU or a network.

## 1. Setup

The cell below imports what the notebook uses and fixes the plotting style. Run it first;
if it fails, check that Matplotlib is installed in the same environment as this kernel.

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (7, 3.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
print("ready")

## 2. A capability score you can compute

The score counts four flags: does the system perceive, decide, adapt, and pursue a goal?
Equal weights are a deliberate simplification — §6 of L01-01 explains how to change them.

In [ ]:
@dataclass(frozen=True)
class System:
    name: str
    perceives: bool
    decides: bool
    adapts: bool
    goal_directed: bool

    @property
    def score(self) -> int:
        return sum([self.perceives, self.decides, self.adapts, self.goal_directed])

    def verdict(self) -> str:
        if self.score <= 1:
            return "automation, not really AI"
        if self.score == 2:
            return "borderline - often sold as AI"
        if self.adapts:
            return "AI: learning system"
        return "AI: hand-written rules"


systems = [
    System("thermostat", True, True, False, True),
    System("spreadsheet SUM", False, True, False, False),
    System("spam filter", True, True, True, True),
    System("chess engine", True, True, False, True),
]
for system in systems:
    print(f"{system.name:16s} score={system.score}  {system.verdict()}")

### Exercise 1 — score your own systems

Add three systems to the list below: a lift button, a voice assistant, and a recommender.
Decide each flag yourself, then run the cell. If two people disagree about a system, find
which flag they disagree about — that is the useful part of the argument.

In [ ]:
mine = [
    # System("lift button", perceives=?, decides=?, adapts=?, goal_directed=?),
]

for system in mine:
    print(f"{system.name:16s} score={system.score}  {system.verdict()}")

<details><summary>Reference answer</summary>

A lift button: perceives ✓ (someone pressed), decides ✓ (which car to send), adapts ✗,
goal-directed ✓ → score 3, hand-written rules. A voice assistant: 4, learning system.
A recommender: 4 unless the ranking is editorial, in which case 2.

</details>

## 3. How fast did the field actually move?

Public milestones are not capability, but they show the rhythm. The gaps below are computed
from the same list used in L01-02, so you can reproduce the lesson's numbers.

In [ ]:
milestones = [
    (1950, "Turing test paper"),
    (1956, "Dartmouth workshop"),
    (1958, "Perceptron"),
    (1966, "ELIZA"),
    (1972, "SHRDLU"),
    (1980, "XCON in industry"),
    (1987, "Backpropagation popularised"),
    (1997, "Deep Blue"),
    (2011, "Watson"),
    (2012, "AlexNet"),
    (2016, "AlphaGo"),
    (2017, "Transformer"),
    (2022, "ChatGPT"),
]

years = np.array([year for year, _ in milestones])
gaps = np.diff(years)
labels = [name for _, name in milestones[1:]]
for label, gap in zip(labels, gaps):
    print(f"{label:32s} +{gap:2d} yr")

In [ ]:
def era_mean(low: int, high: int) -> float:
    chosen = [gap for year, gap in zip(years[1:], gaps) if low <= year < high]
    return float(np.mean(chosen))

eras = [("1950-1979", 1950, 1980), ("1980-2011", 1980, 2012), ("2012+", 2012, 2030)]
for name, low, high in eras:
    print(f"{name:10s} mean gap {era_mean(low, high):.2f} years")

plt.bar(labels, gaps, color="#4a6fa5")
plt.xticks(rotation=75, ha="right")
plt.ylabel("years since previous milestone")
plt.title("Intervals between public AI milestones")
plt.tight_layout()
plt.show()

The middle era is the slowest, and it is the one that contains both AI winters. Notice what
this measure cannot see: internal progress that produced no headline. That is why the chart is
evidence about *public rhythm*, not about capability.

## 4. Containment, checked rather than asserted

AI ⊃ ML ⊃ DL is a claim about sets. Encode it and let Python check membership.

In [ ]:
AI = {"search", "logic", "constraints", "supervised learning", "language models"}
ML = {"supervised learning", "language models"}
DL = {"language models"}

print("ML subset of AI:", ML <= AI)
print("DL subset of ML:", DL <= ML)
print("AI but not ML:", sorted(AI - ML))
print("AI but not DL:", sorted(AI - DL))

If `ML <= AI` were `False`, the taxonomy would be wrong, not Python. Checking containment
in code is a cheap way to catch a muddled diagram.

## 5. Rule versus examples: the same six emails

The rule below is symbolic: a hand-written conjunction. The alternative is 3-nearest-neighbour,
the smallest honest example of learning from data. Both are given eight training emails.

In [ ]:
train = [
    (0.05, 0, "ham"), (0.08, 1, "ham"), (0.02, 0, "ham"), (0.10, 2, "ham"),
    (0.60, 3, "spam"), (0.75, 4, "spam"), (0.55, 2, "spam"), (0.80, 5, "spam"),
]
test = [
    ((0.06, 1), "ham"), ((0.70, 3), "spam"), ((0.35, 2), "spam"),
    ((0.90, 0), "ham"), ((0.90, 1), "spam"), ((0.30, 3), "ham"),
]


def rule(email):
    caps, links = email
    return "spam" if (links >= 2 and caps > 0.3) else "ham"


def knn(email, k=3):
    def distance(other):
        return ((email[0] - other[0]) ** 2 + 0.1 * (email[1] - other[1]) ** 2) ** 0.5

    votes = [label for _, _, label in sorted(train, key=distance)[:k]]
    return "spam" if votes.count("spam") > votes.count("ham") else "ham"


print(f"{'email':10s} {'true':5s} {'rule':5s} {'knn':5s}")
for (caps, links), label in test:
    print(f"({caps:.2f},{links})  {label:5s} {rule((caps, links)):5s} {knn((caps, links)):5s}")

rule_hits = sum(rule(email) == label for email, label in test)
knn_hits = sum(knn(email) == label for email, label in test)
print(f"rule {rule_hits}/{len(test)}   k-NN {knn_hits}/{len(test)}")

### Exercise 2 — find the disagreements

Print only the rows where the two methods disagree, with the true label. Then explain, in a
markdown cell you write yourself, which feature each method is over-weighting.

In [ ]:
for (caps, links), label in test:
    a, b = rule((caps, links)), knn((caps, links))
    if a != b:
        print(f"({caps:.2f},{links}) true={label} rule={a} knn={b}")

<details><summary>Reference answer</summary>

`(0.90,1)` is spam but the rule says ham, because it requires two links. `(0.30,3)` is ham but
k-NN says spam, because links dominate the distance and the nearest spam examples all have
several. The rule over-weights the conjunction of features; k-NN over-weights the link count
relative to capitalisation (the 0.1 scale factor is a modelling choice, not a fact).

</details>

## 6. Calibration: does confidence mean anything?

A model that says "95% sure" should be right about 95% of the time in that bucket. The table
below is one evaluation run of 600 predictions, grouped by stated confidence.

In [ ]:
buckets = [("50-69%", 0.595, 200, 120), ("70-89%", 0.795, 200, 150), ("90-99%", 0.945, 200, 174)]
names = [name for name, _, _, _ in buckets]
stated = np.array([mid for _, mid, _, _ in buckets])
observed = np.array([correct / cases for _, _, cases, correct in buckets])

for name, mid, cases, correct in buckets:
    print(f"{name}: {correct}/{cases} correct, stated {mid:.1%}, observed {correct/cases:.1%}")

ece = float(np.mean(np.abs(observed - stated)))
print(f"expected calibration error: {ece:.3f}")

plt.plot([0, 1], [0, 1], "--", color="grey", label="perfect calibration")
plt.plot(stated, observed, "o-", label="this model")
plt.xlabel("stated confidence")
plt.ylabel("observed accuracy")
plt.legend()
plt.title("Reliability diagram")
plt.tight_layout()
plt.show()

The points sit *below* the diagonal: the model is overconfident in every bucket, and most
overconfident in the top one — precisely where people stop checking. ECE summarises the gap
as one number (0.042 here), but always look at the diagram as well: two models can share an
ECE and fail differently.

### Exercise 3 — a well-calibrated colleague

Construct a second bucket table whose observed accuracies are 0.60, 0.80 and 0.94 for the same
stated midpoints, and compute its ECE. Which model would you deploy behind an "auto-approve
above 90%" rule, and why?

In [ ]:
# your table here: (name, stated midpoint, cases, correct)
better = []

if better:
    stated2 = np.array([mid for _, mid, _, _ in better])
    observed2 = np.array([correct / cases for _, _, cases, correct in better])
    print(f"ECE: {np.mean(np.abs(observed2 - stated2)):.3f}")
else:
    print("fill in `better` to compute its ECE")

<details><summary>Reference answer</summary>

With 200 cases per bucket, correct counts of 120, 160 and 188 give observed 0.60, 0.80, 0.94
and ECE ≈ 0.02. The second model is preferable for auto-approval: in the top bucket its
observed accuracy exceeds the 90% threshold, so the rule's promise is kept. The first model's
87% means 13% of auto-approvals are wrong.

</details>

## 7. Compounding error over a plan

An agent that is right 95% of the time per step is not right 95% of the time on a twenty-step
plan. Independence is an assumption, and even under that generous assumption the arithmetic is
harsh.

In [ ]:
steps = np.arange(1, 31)
for reliability in (0.90, 0.95, 0.99):
    curve = reliability ** steps
    print(f"p={reliability}: 5 steps {reliability**5:.3f}, 10 steps {reliability**10:.3f}, 20 steps {reliability**20:.3f}")

for reliability in (0.90, 0.95, 0.99):
    plt.plot(steps, reliability ** steps, label=f"per-step {reliability}")
plt.xlabel("steps in the plan")
plt.ylabel("probability the whole plan succeeds")
plt.ylim(0, 1)
plt.legend()
plt.title("Compounding error, assuming independent steps")
plt.tight_layout()
plt.show()

### Exercise 4 — verification changes the arithmetic

Suppose a cheap checker catches 90% of failed steps and costs nothing. Model the plan as
succeeding if every step either works or is caught and retried once. Compute the success
probability of a twenty-step plan at per-step reliability 0.95, and compare with 0.358.

In [ ]:
p_step = 0.95
caught, retry_success = 0.90, 0.95
p_effective = p_step + (1 - p_step) * caught * retry_success
print(f"effective per-step success: {p_effective:.4f}")
print(f"20-step plan: {p_effective**20:.3f} vs {p_step**20:.3f} without a checker")

<details><summary>Reference answer</summary>

Effective per-step success is 0.95 + 0.05 × 0.9 × 0.95 = 0.9928, so a twenty-step plan
succeeds with probability ≈ 0.865 instead of 0.358. The lesson generalises: verification is
often a bigger lever than a better model, and it is also cheaper to evaluate.

</details>

## 8. Verify your environment

Run this cell and save its output somewhere you can compare later. Your version and platform
will differ; the module lines should all say `ok`.

In [ ]:
import platform
import sys

print("python:", sys.version.split()[0])
print("platform:", platform.system(), platform.release())
print("executable:", sys.executable)

report = []
for module in ("json", "csv", "math", "random", "statistics", "dataclasses", "itertools", "pathlib"):
    try:
        __import__(module)
        report.append((module, "ok"))
    except ImportError:
        report.append((module, "MISSING"))

for module, status in report:
    print(f"  {module:12s} {status}")
print(f"{sum(status == 'ok' for _, status in report)}/{len(report)} modules available")

## 9. What you should be able to do now

- Score any described system and defend the score flag by flag.
- Say what a milestone-gap chart does and does not measure.
- Compare a rule with a nearest-neighbour classifier and explain the disagreement.
- Compute ECE from a bucket table and interpret a reliability diagram.
- Compute compounded failure probability, and show what verification buys.

Next: attempt [LAB-01-01](../labs/lab-01-01-claim-audit.md), which applies these tools to a
real capability claim, and [LAB-01-02](../labs/lab-01-02-turing-test-conversation.md), which
makes conversational fluency measurable.